# Imports

In [10]:
import os
import urllib.request
import zipfile
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm


# Local Directory Setup

In [2]:

# Local Directory Setup 
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Local Data Directory set to: {DATA_DIR}")

# CRITICAL: GitHub Security (.gitignore)
gitignore_path = os.path.join(os.getcwd(), '.gitignore')
with open(gitignore_path, 'a+') as f:
    f.seek(0)
    content = f.read()
    if 'TFE_Data/' not in content:
        f.write("\n# Ignore Dataset Folder\nTFE_Data/\n")
        f.write("__pycache__/\n*.npy\n*.zip\n")
print(".gitignore security is active.")


Local Data Directory set to: /home/aysel/tfe/TFE_Data
.gitignore security is active.


In [3]:
def download_and_extract(url, extract_to): 
    """Downloads a zip file from the specified URL and extracts it to the given directory."""
    
    os.makedirs(extract_to, exist_ok=True)
    zip_path = os.path.join(extract_to, "temp.zip")
    
    if not os.listdir(extract_to): 
        print(f"Downloading from {url}...")
        urllib.request.urlretrieve(url, zip_path) # Download the zip file
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to) # Extract the contents
        os.remove(zip_path) # Clean up the zip file
        print("Extraction complete.")
    else:
        print(f"Files already exist in {extract_to}. Skipping download.")

# Flikr8k

In [7]:
def build_flickr8k():
    print("\n--- BUILDING FLICKR8K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data) # Create a flat DataFrame with one row per caption
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index() # Group captions by image
    df.rename(columns={'caption': 'captions'}, inplace=True) # Rename column to 'captions' for clarity
    
    save_path = os.path.join(DATA_DIR, 'df_Flickr8k.pkl') # Save the DataFrame as a pickle file
    df.to_pickle(save_path) 
    
    print(f"Flickr8k Ready: {len(df)} images secured.")
    return "Flickr8k"

In [8]:
def build_flickr8k_subset(subset_size=2000):
    print("\n--- BUILDING FLICKR8K SUBSET ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr8k")
    img_dir = os.path.join(dataset_dir, "Images")
    txt_dir = os.path.join(dataset_dir, "Text")
    
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip", img_dir)
    download_and_extract("https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip", txt_dir)
    
    actual_images_dir = os.path.join(img_dir, "Flicker8k_Dataset")
    token_path = os.path.join(txt_dir, "Flickr8k.token.txt")
    data = []

    print("Verifying physical files and syncing captions...")
    with open(token_path, 'r') as f:
        for line in f.read().splitlines():
            if not line: continue
            parts = line.split('\t')
            if len(parts) == 2:
                img_name = parts[0].split('#')[0]
                full_path = os.path.join(actual_images_dir, img_name)
                
                if os.path.exists(full_path):
                    data.append({
                        "image_name": img_name, 
                        "image_path": full_path, 
                        "caption": parts[1].strip()
                    })

    df_flat = pd.DataFrame(data) # Create a flat DataFrame with one row per caption
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index() # Group captions by image
    df.rename(columns={'caption': 'captions'}, inplace=True) # Rename column to 'captions' for clarity
    
    subset_df = df.head(subset_size).copy() # Select a subset of the data
    save_path = os.path.join(DATA_DIR, 'subset_df_Flickr8k.pkl') # Save the subset DataFrame as a pickle file
    subset_df.to_pickle(save_path) 
    
    print(f" Flickr8k Subset Ready: {len(subset_df)} images secured.")
    return "Flickr8k Subset"

# Flicker30k

In [1]:
import os
import shutil
import kagglehub

os.environ["KAGGLE_API_TOKEN"] = "KGAT_8d2c016f8899a1eab14c99441d2bf6fe"

DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
flickr30k_dir = os.path.join(DATA_DIR, 'Flickr30k')
os.makedirs(flickr30k_dir, exist_ok=True)

print("Downloading Flickr30k(about 8 Go)...")

dataset_cache_path = kagglehub.dataset_download("hsankesara/flickr-image-dataset")

print(f"Download comlpeted and temp files found in : {dataset_cache_path}")
print("Copying data to TFE_Data.")

shutil.copytree(dataset_cache_path, flickr30k_dir, dirs_exist_ok=True)

print(f"Flickr30k is ready in : {flickr30k_dir}")

100%|██████████| 8.16G/8.16G [03:52<00:00, 37.7MB/s]  

Extracting files...


Download comlpeted and temp files found in : /home/aysel/.cache/kagglehub/datasets/hsankesara/flickr-image-dataset/versions/1
Copying data to TFE_Data.
Flickr30k is ready in : /home/aysel/tfe/TFE_Data/Flickr30k


In [12]:
def build_flickr30k():
    print("\n--- BUILDING FLICKR30K ---")
    dataset_dir = os.path.join(DATA_DIR, "Flickr30k")
    img_dir = os.path.join(dataset_dir, "Images")
    csv_path = os.path.join(dataset_dir, "results.csv")
    
    if not os.path.exists(csv_path) or not os.path.exists(img_dir):
        print(f"⚠️ Fichiers introuvables dans {dataset_dir}")
        return None

    print("Lecture du fichier texte...")
    # 1. On tente avec le séparateur '|', sinon avec ','
    df_cap = pd.read_csv(csv_path, sep='|', on_bad_lines='skip') 
    if len(df_cap.columns) < 2:
        df_cap = pd.read_csv(csv_path, sep=',', on_bad_lines='skip')
        
    # Nettoyer les noms de colonnes
    df_cap.columns = [str(col).strip() for col in df_cap.columns] 
    
    # DIAGNOSTIC
    if 'image_name' not in df_cap.columns:
        print(f"❌ ERREUR : La colonne 'image_name' n'existe pas. Colonnes trouvées : {df_cap.columns.tolist()}")
        return None

    sample_name = str(df_cap.iloc[0]['image_name']).strip()
    sample_path = os.path.join(img_dir, sample_name)
    print(f"🔍 DIAGNOSTIC : On cherche par exemple l'image '{sample_name}'")
    print(f"📍 Chemin testé : {sample_path}")
    print(f"👀 Existe-t-elle physiquement ? : {'OUI ✅' if os.path.exists(sample_path) else 'NON ❌'}")
    
    data = []
    for index, row in df_cap.iterrows():
        img_name = str(row['image_name']).strip()
        caption = str(row.get('comment', '')).strip() # Gère si la colonne s'appelle un peu différemment
        full_path = os.path.join(img_dir, img_name)
        
        # On ne garde que si l'image existe physiquement
        if os.path.exists(full_path) and caption != 'nan' and caption != '':
            data.append({
                "image_name": img_name, 
                "image_path": full_path, 
                "caption": caption
            })

    # 2. Vérification de sécurité CRITIQUE
    if len(data) == 0:
        print("\n❌ ERREUR FATALE : 0 image trouvée ! Vos images ne sont pas directement dans le dossier 'Images'.")
        print("Veuillez vérifier manuellement où se trouvent vos fichiers .jpg dans l'arborescence de votre serveur.")
        return None

    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATA_DIR, 'df_Flickr30k.pkl')
    df.to_pickle(save_path) 
    
    print(f"🎉 Flickr30k Ready: {len(df)} images secured et sauvegardées dans {save_path}")
    return "Flickr30k"

# Conceptual Captions

In [16]:
def download_image_task(item):
    """Fonction exécutée par les threads pour télécharger une image."""
    index, url, caption, img_dir = item
    img_name = f"gcc_50k_{index}.jpg"
    full_path = os.path.join(img_dir, img_name)
    
    # Si l'image existe déjà, on valide direct
    if os.path.exists(full_path):
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
        
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        response = requests.get(url, headers=headers, timeout=4)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        # Compression JPEG pour économiser de la place
        image.save(full_path, format="JPEG", quality=85)
        return {"success": True, "image_name": img_name, "image_path": full_path, "caption": caption}
    except:
        return {"success": False}



In [17]:
def build_conceptual_captions_50k(target_size=50000, max_workers=32):
    """Télécharge exactement 50 000 images valides du dataset CC."""
    print(f"\n--- BUILDING CONCEPTUAL CAPTIONS ({target_size} images) ---")
    dataset_dir = os.path.join(DATA_DIR, "ConceptualCaptions")
    img_dir = os.path.join(dataset_dir, "Images")
    os.makedirs(img_dir, exist_ok=True)
    
    try:
        from datasets import load_dataset
    except ImportError:
        import subprocess
        subprocess.check_call(["pip", "install", "datasets"])
        from datasets import load_dataset

    print("1. Chargement des métadonnées HuggingFace...")
    # On charge le dataset
    ds = load_dataset("conceptual_captions", split="train")
    
    # On prend les 100 000 premiers pour être sûr d'en avoir 50 000 valides
    print("2. Sélection d'un pool de 100 000 URLs pour pallier aux liens morts...")
    ds_subset = ds.select(range(100000))
    tasks = [(i, url, cap, img_dir) for i, (url, cap) in enumerate(zip(ds_subset['image_url'], ds_subset['caption']))]
        
    data = []
    successful_downloads = 0
    
    print(f"3. Lancement du téléchargement avec {max_workers} threads...")
    
    # Lancement du multithreading
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # On soumet toutes les tâches
        futures = {executor.submit(download_image_task, task): task for task in tasks}
        
        # tqdm pour la barre de progression (limitée à target_size)
        pbar = tqdm(total=target_size, desc="Images valides téléchargées")
        
        for future in as_completed(futures):
            res = future.result()
            if res["success"]:
                data.append({
                    "image_name": res["image_name"],
                    "image_path": res["image_path"],
                    "caption": res["caption"]
                })
                successful_downloads += 1
                pbar.update(1)
                
            # On coupe TOUT dès qu'on atteint les 50 000 !
            if successful_downloads >= target_size:
                print(f"\n🎯 Objectif de {target_size} atteint ! Arrêt des téléchargements en cours...")
                # Annuler les tâches restantes
                for f in futures:
                    f.cancel()
                break
                
        pbar.close()

    print("\n4. Formatage et sauvegarde du DataFrame...")
    df_flat = pd.DataFrame(data)
    df = df_flat.groupby(['image_name', 'image_path'])['caption'].apply(list).reset_index()
    df.rename(columns={'caption': 'captions'}, inplace=True)
    
    save_path = os.path.join(DATA_DIR, 'df_ConceptualCaptions.pkl')
    df.to_pickle(save_path) 
    
    print(f"Le dataset de {len(df)} images est prêt dans {save_path}")
    return "ConceptualCaptions"

# Execution of Downloading Flikr8k

In [13]:

# --- Exécution ---
if __name__ == "__main__":
    #build_flickr8k()
    build_flickr30k()

    #build_conceptual_captions_50k(target_size=50000, max_workers=32)


--- BUILDING FLICKR30K ---
Lecture du fichier texte...
🔍 DIAGNOSTIC : On cherche par exemple l'image '1000092795.jpg'
📍 Chemin testé : /home/aysel/tfe/TFE_Data/Flickr30k/Images/1000092795.jpg
👀 Existe-t-elle physiquement ? : OUI ✅
🎉 Flickr30k Ready: 31783 images secured et sauvegardées dans /home/aysel/tfe/TFE_Data/df_Flickr30k.pkl
